In [1]:
!pip install -U flax optax icecream jax[cuda12] jaxlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 141.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.8/175.8 MB 15.2 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: jax-cuda12-pjrt
    Found existing installation: jax-cuda12-pjrt 0.7.2
    Uninstalling jax-cuda12-pjrt-0.7.2:
      Successfully uninstalled jax-cuda12-pjrt-0.7.2
  Attempting uninstall: jax-cuda12-plugin
    Found existing installation: jax-cuda12-plugin 0.7.2
    Uninstalling jax-cuda12-plugin-0.7.2:
      Successfully uninstalled jax-cuda12-plugin-0.7.2


In [2]:
!git clone https://github.com/RaameshB/JAX-Mambas.git

fatal: destination path 'JAX-Mambas' already exists and is not an empty directory.


In [3]:
!cd JAX-Mambas && git switch dev && git pull

Already on 'dev'
Your branch is up to date with 'origin/dev'.
Already up to date.


In [4]:
import sys
sys.path.append("/content/JAX-Mambas")

In [5]:
import jax
from jax import numpy as jnp
from jax import random
from functools import partial
from flax import nnx
from mamba.mamba1 import Mamba
import optax
from icecream import ic

In [6]:
def generate_induction_heads(rng_key, seq_len=256, vocab_size=16):
    special_key, content_key = random.split(rng_key)
    special_token = jnp.array([vocab_size-1])
    sequence = jnp.concat((random.randint(content_key, (seq_len-1,), minval=0, maxval=vocab_size-1), special_token))
    special_loc = random.randint(special_key, (1,), minval=0, maxval=seq_len-2)
    sequence_with_key = sequence.at[special_loc[...]].set(jnp.array(special_token))
    value = sequence_with_key[special_loc[...]+1]
    return sequence_with_key, value[0]


def create_batch(key, bsz, seq_len=256, vocab_size=16):
    induction_heads_batch, values = jax.vmap(
        partial(generate_induction_heads, seq_len=seq_len, vocab_size=vocab_size)
    )(random.split(key, bsz))
    one_hot_y = jax.nn.one_hot(values, vocab_size)
    return induction_heads_batch, one_hot_y

In [7]:
class MambaInductionHeads(nnx.Module):
    def __init__(self, rngs, vocab_size=16, D=64, expand=2, num_layers=2):
        self.embed = nnx.Embed(num_embeddings=vocab_size, features=D, rngs=rngs)
        self.mambas = nnx.List([Mamba(D=D, expand=expand, rngs=rngs) for _ in range(num_layers)])
        self.proj_down = nnx.Linear(in_features=D, out_features=vocab_size, rngs=rngs)
    def __call__(self, x):
        hidden = self.embed(x)
        for mamba in self.mambas:
            hidden = mamba(hidden)
        return self.proj_down(hidden)

In [8]:
lr = 1e-3
bsz = 32
train_steps = 50

In [9]:
rngs = nnx.Rngs(0)
model = MambaInductionHeads(rngs=rngs)
graphdef, params = nnx.split(model, nnx.Param)
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

In [10]:
@nnx.jit
def train_step(rngs, graphdef, params, opt_state):
    def compute_loss(params, inputs, labels):
        model = nnx.merge(graphdef, params)
        logits = model(inputs)[:,-1]
        loss = jnp.mean(optax.losses.safe_softmax_cross_entropy(logits, labels))
        return loss
    batch_x, batch_y = create_batch(rngs.inputs(), bsz=bsz)
    loss, grads = jax.value_and_grad(compute_loss)(params, batch_x, batch_y)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

In [11]:

for step in range(train_steps):
    params, opt_state, loss = train_step(rngs, graphdef, params, opt_state)
    print(f'Step: {step}, Loss: {loss}')

AssertionError: must override